# Exercise 07 — Instruction Tuning and Preference Alignment

The model you prompted in notebook 1 started out as a **base model**, pretrained like GPT-2 to continue text. A base model has never been taught to *answer* a question, to stop when it is done, or to prefer good answers over bad ones. Two more training stages turn it into an assistant:

1. **Supervised fine-tuning (SFT)**, also called *instruction tuning*: train on examples of prompts with good responses. The loss is the same next-token prediction as in pretraining, but on curated conversations.
2. **Preference alignment**: train on *pairs* of responses to the same prompt, one preferred and one rejected. The classic method is RLHF, reinforcement learning from human feedback ([Ouyang et al., 2022](https://arxiv.org/abs/2203.02155), the InstructGPT paper behind ChatGPT). This notebook uses a simpler alternative that works without reinforcement learning, **Direct Preference Optimization** (DPO; [Rafailov et al., 2023](https://arxiv.org/abs/2305.18290)).

You go through both stages on a small scale. You start from the *base* model **Qwen2.5-0.5B** and teach it to solve GSM8K math problems step by step in a chat format, first by fine-tuning all of its weights and then with **LoRA**, a parameter-efficient method. Finally, you align the LoRA model with DPO on pairs of correct and incorrect solutions. You use [TRL](https://huggingface.co/docs/trl/index), Hugging Face's library for training language models after pretraining, and [PEFT](https://huggingface.co/docs/peft/index) for LoRA.

## What you will do
1. **The base model**: what a pretrained model does with a question.
2. **The data**: GSM8K as prompt–completion conversations, and the chat template.
3. **Measuring accuracy**: the base model on 100 test problems.
4. **Supervised fine-tuning**: all 494 M parameters, with TRL's `SFTTrainer`.
5. **LoRA**: counting its parameters, and fine-tuning with 2% of them.
6. **DPO**: preference pairs, the DPO objective, and aligning the LoRA model.
7. **Multiple-choice questions**: to check your understanding.

## How to work through it
- Run the cells **in order** and fill in **every `# TODO`**.
- Tasks are numbered (**1.1**, **1.2**, …). Tasks that say *Your answer here* want a short written answer, not code.
- Most implementation tasks are followed by a **✅ Check** cell that verifies your work automatically. Run it and make sure it passes before you move on.

> **Heads up: this notebook needs a GPU.** Training a 0.5 B parameter model on a CPU would take many hours. Use a free GPU on Google Colab (*Runtime → Change runtime type → T4 GPU*) or Kaggle (*Settings → Accelerator → GPU T4*); see `COMPUTE_RESOURCES.md`. On a T4, each of the three training runs takes roughly 5–15 minutes and each evaluation 1–3 minutes, so plan about an hour for the whole notebook. Start a run, then read ahead while it trains.
>
> The notebook downloads Qwen2.5-0.5B (~1 GB), GSM8K (~5 MB) and Math-Step-DPO-10K (~12 MB). On Colab or Kaggle, first run the install cell below. Locally, `uv sync` has already installed everything.

In [ ]:
# Only on Colab or Kaggle: install TRL and PEFT (remove the # and run this cell once, before anything else).
# On Kaggle, run `!pip install uv` first. If you are asked to restart the session afterwards, do so.
# !uv pip install --system "trl>=1.14" "peft>=0.21"

In [ ]:
%matplotlib inline
import gc
import os
import re
import time

import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from peft import AutoPeftModelForCausalLM, LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOConfig, DPOTrainer, SFTConfig, SFTTrainer

torch.manual_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")
if device.type == "cuda":
    print(torch.cuda.get_device_name())
else:
    print("⚠️ No GPU found. Parts 2 and 5.1 work on a CPU, but training and evaluation would take hours.")

# Mixed precision: bf16 on recent GPUs (A100, L4, H100, ...), fp16 on older ones such as the T4
USE_BF16 = device.type == "cuda" and torch.cuda.is_bf16_supported(including_emulation=False)
USE_FP16 = device.type == "cuda" and not USE_BF16

### Settings

The sizes below are chosen so that every run fits on a free T4 GPU in reasonable time. Leave them as they are for your first run.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B"  # the *base* model, before any instruction tuning
N_SFT = 2000         # GSM8K training problems for supervised fine-tuning (out of 7,473)
N_DPO = 1000         # preference pairs for DPO
N_EVAL = 100         # GSM8K test problems for evaluation (out of 1,319)
MAX_NEW_TOKENS = 256  # maximum length of a generated solution

## 1. The base model

[Qwen2.5-0.5B](https://huggingface.co/Qwen/Qwen2.5-0.5B) is the smallest model of Alibaba's Qwen2.5 family ([Qwen Team, 2024](https://arxiv.org/abs/2412.15115)). It is a decoder-only Transformer like your mini GPT, with 24 layers and 494 M parameters, pretrained on 18 trillion tokens of web text, code and math. The family also includes an *-Instruct* version that has gone through SFT and preference alignment; we start from the base model and do those steps ourselves.

The model is loaded in 32-bit precision. During training, the trainers compute in 16-bit (*mixed precision*), which is faster and needs less memory, while keeping a 32-bit copy of the weights for the updates.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"  # for batched generation, see below


def load_base_model():
    # A fresh copy of the pre-trained model
    return AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(device)


model = load_base_model()
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

First, give the base model a GSM8K question as plain text and let it continue, exactly as your mini GPT continued Shakespeare.

In [ ]:
question_text = (
    "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. "
    "How many clips did Natalia sell altogether in April and May?"
)

inputs = tokenizer(question_text, return_tensors="pt").to(device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=150, do_sample=False, pad_token_id=tokenizer.pad_token_id)
print(tokenizer.decode(output[0][inputs["input_ids"].shape[1]:]))

Chat models see conversations in a fixed format, the **chat template**. Qwen uses the *ChatML* format: every message is wrapped in `<|im_start|>role` … `<|im_end|>`, and the prompt ends with `<|im_start|>assistant`, the cue for the model to answer. `tokenizer.apply_chat_template` builds this text from a list of messages, like the ones you sent to the API in notebook 1.

The function `generate` below uses the chat template to get the model's replies to a list of questions. Two details:
- Prompts of different lengths are **padded on the left**. The model continues from the *last* position of each sequence, so all prompts have to end at the same position.
- Generation stops at `<|im_end|>`, the end of the assistant's message, or after `MAX_NEW_TOKENS` tokens.

In [ ]:
IM_END = tokenizer.convert_tokens_to_ids("<|im_end|>")


@torch.no_grad()
def generate(model, questions, batch_size=16, max_new_tokens=MAX_NEW_TOKENS):
    # The model's greedy reply to every question, using the chat template
    model.eval()
    replies = []
    for start in range(0, len(questions), batch_size):
        prompts = [
            tokenizer.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)
            for q in questions[start : start + batch_size]
        ]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            eos_token_id=[IM_END, tokenizer.eos_token_id], pad_token_id=tokenizer.pad_token_id,
        )
        replies += tokenizer.batch_decode(outputs[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return replies


print(tokenizer.apply_chat_template([{"role": "user", "content": question_text}], tokenize=False, add_generation_prompt=True))
print("=" * 60)
print(generate(model, [question_text])[0])

**1.1 What does the base model do with the question**, as plain text and in the chat format? What is missing for it to be a useful assistant?

---

*Your answer here:*

---

## 2. The data

### Prompt–completion conversations

You met GSM8K in notebook 1. For SFT we turn every problem into a **conversation**: the question is a `user` message, and the worked solution is the `assistant` message the model should learn to produce. TRL calls this the *prompt–completion* format:

```python
{
    "prompt":     [{"role": "user",      "content": "Natalia sold clips to 48 ..."}],
    "completion": [{"role": "assistant", "content": "Natalia sold 48/2 = 24 clips in May.\n...\nThe answer is: 72"}],
}
```

We clean up the reference solutions a little: the calculator annotations (`<<48/2=24>>`) are removed, and the final line `#### 72` becomes `The answer is: 72`, the format of the preference data in Part 6.

**2.1 Complete `to_prompt_completion`.**

Hint: as in notebook 1, split the answer at `####` and remove the annotations with `re.sub(r"<<[^>]*>>", "", text)`.

In [ ]:
gsm8k = load_dataset("openai/gsm8k", "main")


def to_prompt_completion(example):
    # A GSM8K example -> {"prompt": [user message], "completion": [assistant message]}
    reasoning = ...   # TODO: the text before "####", without calculator annotations and surrounding whitespace
    answer = ...      # TODO: the text after "####", without surrounding whitespace
    completion = f"{reasoning}\nThe answer is: {answer}"
    return {
        "prompt": ...,      # TODO: a list with one user message, the question
        "completion": ...,  # TODO: a list with one assistant message, the completion
    }


to_prompt_completion(gsm8k["train"][0])

In [ ]:
# ✅ Check your to_prompt_completion
converted = to_prompt_completion(gsm8k["train"][0])
assert converted["prompt"] == [{"role": "user", "content": gsm8k["train"][0]["question"]}], "the prompt should be one user message with the question"
assert converted["completion"] == [{
    "role": "assistant",
    "content": "Natalia sold 48/2 = 24 clips in May.\nNatalia sold 48+24 = 72 clips altogether in April and May.\nThe answer is: 72",
}], f"unexpected completion: {converted['completion']}"
print("Looks good ✅")

In [ ]:
sft_data = gsm8k["train"].shuffle(seed=0).select(range(N_SFT)).map(to_prompt_completion, remove_columns=["question", "answer"])
eval_problems = gsm8k["test"].shuffle(seed=0).select(range(N_EVAL))
print(sft_data)

### What the model actually sees

The trainer turns every conversation into a single text with the chat template, tokenizes it, and trains the model to predict the next token, as in pretraining. With the prompt–completion format, the loss is computed **only on the completion**: the tokens of the prompt get the label `-100`, the value that PyTorch's cross-entropy ignores (the `ignore_index` you used for padding in week 5).

In [ ]:
example = sft_data[0]
print(tokenizer.apply_chat_template(example["prompt"] + example["completion"], tokenize=False))

**2.2 Two questions about the training text.**
1. Why is the loss computed only on the completion and not on the question?
2. The completion ends with `<|im_end|>`. Why is it important that the model is trained to predict this token?

---

*Your answer here:*

---

## 3. Measuring accuracy

To compare the models, we let each of them solve the same `N_EVAL` test problems with greedy decoding and check the final answers. `extract_answer` works like the one you wrote in notebook 1, but looks for `The answer is:`; if that phrase is missing, it falls back to the last number in the reply.

In [ ]:
NUMBER = r"-?\d[\d,]*(?:\.\d+)?"


def extract_answer(reply):
    # The final answer in a reply as a float, or None
    if "The answer is:" in reply:
        numbers = re.findall(NUMBER, reply.rsplit("The answer is:", 1)[-1])
        number = numbers[0] if numbers else None
    else:
        numbers = re.findall(NUMBER, reply)
        number = numbers[-1] if numbers else None
    return None if number is None else float(number.replace(",", ""))


def evaluate(model, name):
    # Accuracy on the evaluation problems; also returns the replies
    start = time.time()
    replies = generate(model, list(eval_problems["question"]))
    golds = [float(answer.split("####")[-1].strip().replace(",", "")) for answer in eval_problems["answer"]]
    preds = [extract_answer(reply) for reply in replies]
    accuracy = sum(p is not None and abs(p - g) < 1e-6 for p, g in zip(preds, golds)) / len(golds)
    formatted = sum("The answer is:" in reply for reply in replies) / len(replies)
    print(f"{name}: accuracy {accuracy:.0%}, 'The answer is:' in {formatted:.0%} of the replies   ({time.time() - start:.0f} s)")
    return accuracy, replies


scores = {}

**3.1 Evaluate the base model.** This takes 1–3 minutes on a T4. Then look at a few of its replies.

In [ ]:
scores["base"], base_replies = evaluate(model, "base model")
for reply in base_replies[:2]:
    print("-" * 60)
    print(reply)

**3.2 How well does the base model do,** and what goes wrong in the replies you looked at? Is a wrong answer always a *reasoning* error?

---

*Your answer here:*

---

## 4. Supervised fine-tuning

SFT is next-token prediction on the completions, with the pre-trained model as the starting point. All 494 M parameters are updated, just like DistilBERT in notebook 0. TRL's [`SFTTrainer`](https://huggingface.co/docs/trl/sft_trainer) is a `Trainer` that knows about conversations: it applies the chat template, tokenizes, masks the prompt tokens and pads the batches.

The training settings go into an `SFTConfig`, a `TrainingArguments` with some extra options. Read through them:

In [ ]:
sft_config = SFTConfig(
    output_dir="qwen-gsm8k-sft",
    num_train_epochs=1,               # one pass over the N_SFT conversations
    per_device_train_batch_size=8,    # conversations per forward/backward pass
    gradient_accumulation_steps=2,    # add up the gradients of 2 passes before each update: an effective batch of 16
    learning_rate=1e-5,               # small, as in notebook 0: we only adapt the pre-trained weights
    lr_scheduler_type="cosine",
    warmup_steps=0.05,                # warm up over the first 5% of the steps
    max_length=512,                   # longer conversations are truncated (GSM8K conversations are much shorter)
    gradient_checkpointing=True,      # recompute activations in the backward pass: ~20% slower, much less memory
    bf16=USE_BF16,
    fp16=USE_FP16,
    logging_steps=10,
    save_strategy="no",               # do not save checkpoints (each would be ~2 GB)
    report_to="none",
    seed=0,
)

**4.1 Create the `SFTTrainer`** with the model, the configuration (`args`), the training data and the tokenizer (`processing_class`), and train. On a T4 this takes roughly 5–15 minutes.

In [ ]:
trainer = ...  # TODO: an SFTTrainer

start = time.time()
trainer.train()
print(f"Training took {time.time() - start:.0f} s")

In [ ]:
def plot_loss(trainer, title):
    history = [(log["step"], log["loss"]) for log in trainer.state.log_history if "loss" in log]
    steps, losses = zip(*history)
    plt.figure(figsize=(8, 3))
    plt.plot(steps, losses)
    plt.xlabel("training step")
    plt.ylabel("training loss")
    plt.title(title)
    plt.show()


plot_loss(trainer, "full SFT")
scores["full SFT"], sft_replies = evaluate(trainer.model, "full SFT")
print(sft_replies[0])

In [ ]:
# ✅ Check the fine-tuned model
formatted = sum("The answer is:" in reply for reply in sft_replies) / len(sft_replies)
assert formatted > 0.8, f"after SFT, most replies should end with 'The answer is: ...', got {formatted:.0%}"
print("Looks good ✅")

**4.2 Compare the base model and the SFT model.** What has changed in the replies, and how much has the accuracy changed? What do you think SFT on 2,000 examples mainly teaches the model: new math skills, or something else?

---

*Your answer here:*

---

Full fine-tuning keeps the weights, their gradients and the optimiser state in GPU memory. Before the next part we free that memory.

In [ ]:
del trainer, model
gc.collect()
torch.cuda.empty_cache()

## 5. LoRA: fine-tuning with few parameters

Full fine-tuning has two problems at scale. Every task gets its own copy of all the weights, and training needs memory for the weights, their gradients and the optimiser state, about 16 bytes per parameter with AdamW in 32-bit precision. That is over 100 GB for a 7 B model.

**LoRA** (Low-Rank Adaptation; [Hu et al., 2021](https://arxiv.org/abs/2106.09685)) freezes all pre-trained weights and learns a small *change* to some weight matrices instead. For a frozen linear layer with weight $W \in \mathbb{R}^{d_\text{out} \times d_\text{in}}$, LoRA adds two small trainable matrices $A \in \mathbb{R}^{r \times d_\text{in}}$ and $B \in \mathbb{R}^{d_\text{out} \times r}$, with a small **rank** $r$ (typically 8–64):

$$h = W x + \frac{\alpha}{r} \, B A x$$

- $BA$ is a $d_\text{out} \times d_\text{in}$ matrix of rank at most $r$: the update to $W$ is assumed to be *low-rank*.
- $B$ starts at zero, so at the start of training the model is exactly the pre-trained model.
- $\alpha$ is a scaling factor, so that you can change $r$ without retuning the learning rate.
- After training, $\frac{\alpha}{r} BA$ can be added to $W$ (*merged*), and the model is as fast as before.

**5.1 Count the trainable parameters of LoRA.** Complete `lora_parameter_count`, which goes through all linear layers of the model whose name ends with one of the `target_modules` and adds up the parameters of $A$ and $B$.

Hints:
- `model.named_modules()` yields `(name, module)` pairs, for example `("model.layers.0.self_attn.q_proj", Linear(...))`.
- An `nn.Linear` has the attributes `in_features` and `out_features`.

In [ ]:
model = load_base_model()


def lora_parameter_count(model, r, target_modules):
    # Number of parameters LoRA with rank r adds to the linear layers called one of target_modules
    total = 0
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear) and name.split(".")[-1] in target_modules:
            total += ...  # TODO: the number of parameters of A and B for this layer
    return total


ATTENTION = ["q_proj", "k_proj", "v_proj", "o_proj"]
MLP = ["gate_proj", "up_proj", "down_proj"]
n_params = sum(p.numel() for p in model.parameters())
for name, targets in [("attention only", ATTENTION), ("attention + MLP", ATTENTION + MLP)]:
    for r in [8, 16, 64]:
        n_lora = lora_parameter_count(model, r, targets)
        print(f"{name:<16} r={r:<3} {n_lora:>11,} parameters  ({n_lora / n_params:.2%} of the model)")

In [ ]:
# ✅ Check your lora_parameter_count
assert lora_parameter_count(model, 1, ["q_proj"]) == 24 * (896 + 896), "q_proj is a 896 x 896 matrix in each of the 24 layers: A has r x 896 and B 896 x r parameters"
assert lora_parameter_count(model, 16, ATTENTION) == 2_162_688
assert lora_parameter_count(model, 16, ATTENTION + MLP) == 8_798_208
print("Looks good ✅")

**5.2 Estimate the memory for training.** With AdamW and 32-bit precision, every *trained* parameter needs about 16 bytes (4 for the weight, 4 for its gradient, 8 for Adam's two running averages); a *frozen* parameter needs only its 4 bytes. Estimate the memory for full fine-tuning of Qwen2.5-0.5B, and for LoRA with $r = 16$ on attention and MLP (ignoring activations). How large is the saved result in each case?

---

*Your answer here:*

---

### Training with LoRA

We apply LoRA to all attention and MLP projections with $r = 16$ and $\alpha = 32$. Everything else stays as before; only the learning rate is higher. The LoRA matrices start from zero, and LoRA is less prone to forgetting, so they can be trained with larger steps.

**5.3 Create an `SFTTrainer` that trains with LoRA.** It is the same as in 4.1, plus one argument: pass the `LoraConfig` as `peft_config`. The trainer then freezes the model and inserts the LoRA matrices.

In [ ]:
lora_config = LoraConfig(
    r=16,                                # rank of the update BA
    lora_alpha=32,                       # scaling factor alpha (the update is scaled by alpha / r = 2)
    lora_dropout=0.05,                   # dropout on the LoRA path
    target_modules=ATTENTION + MLP,      # the linear layers that get LoRA matrices
    task_type="CAUSAL_LM",
)

lora_sft_config = SFTConfig(
    output_dir="qwen-gsm8k-lora",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,                  # 20 times higher than for full fine-tuning
    lr_scheduler_type="cosine",
    warmup_steps=0.05,
    max_length=512,
    gradient_checkpointing=True,
    bf16=USE_BF16,
    fp16=USE_FP16,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    seed=0,
)

trainer = ...  # TODO: an SFTTrainer with LoRA
trainer.model.print_trainable_parameters()

In [ ]:
# ✅ Check your LoRA trainer
n_trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
assert n_trainable == 8_798_208, f"only the LoRA matrices should be trainable (8,798,208 parameters), got {n_trainable:,}"
print("Looks good ✅")

Now train (roughly 5–15 minutes on a T4), evaluate, and save the result. `save_model` saves only the LoRA matrices, the **adapter**, and not the frozen base model.

In [ ]:
start = time.time()
trainer.train()
print(f"Training took {time.time() - start:.0f} s")

plot_loss(trainer, "LoRA SFT")
scores["LoRA SFT"], lora_replies = evaluate(trainer.model, "LoRA SFT")

LORA_DIR = "qwen-gsm8k-lora"
trainer.save_model(LORA_DIR)
size = sum(os.path.getsize(os.path.join(LORA_DIR, f)) for f in os.listdir(LORA_DIR) if f.endswith(".safetensors"))
print(f"The saved adapter takes {size / 1e6:.0f} MB")

**5.4 Compare LoRA with full fine-tuning**: accuracy, training time, and the size of what you saved. Why can a change of rank 16 be enough to adapt a 494 M parameter model?

---

*Your answer here:*

---

In [ ]:
del trainer, model
gc.collect()
torch.cuda.empty_cache()

## 6. Preference alignment with DPO

SFT shows the model *one* good answer per question. Often it is easier to say which of two answers is **better** than to write the perfect one, and comparisons also tell the model what to *avoid*. Preference data consists of triples: a prompt $x$, a preferred (*chosen*) response $y_w$ and a *rejected* response $y_l$.

**RLHF** uses such data in two steps: first it trains a reward model to score responses, then it optimises the language model with reinforcement learning to get high rewards while staying close to the SFT model. **DPO** showed that the same goal can be reached with a single, simple loss on the preference pairs, without a reward model and without reinforcement learning:

$$\mathcal{L}_\text{DPO} = -\log \sigma\Big(\beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_\text{ref}(y_w \mid x)} \;-\; \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_\text{ref}(y_l \mid x)}\Big)$$

- $\pi_\theta$ is the model being trained, and $\pi_\text{ref}$ a frozen copy of the starting point, here the LoRA SFT model.
- $\beta \log \frac{\pi_\theta(y \mid x)}{\pi_\text{ref}(y \mid x)}$ is called the **implicit reward** of response $y$: how much more likely the model has made $y$ compared with the reference.
- The loss pushes the reward of the chosen response above that of the rejected one. It does not matter whether the probabilities of both go up or down, only their difference.
- $\beta$ controls how far the model may move away from the reference: a larger $\beta$ means a stronger pull towards the reference model.

### The preference data

[Math-Step-DPO-10K](https://huggingface.co/datasets/xinlai/Math-Step-DPO-10K) ([Lai et al., 2024](https://arxiv.org/abs/2406.18629)) contains about 10,800 math problems, each with a correct and an incorrect step-by-step solution. The two solutions share their first steps (`initial_reason_steps`), and then the chosen solution continues correctly while the rejected one makes a mistake. The problems come from GSM8K, from the harder MATH dataset and from AQuA, mostly in rephrased or augmented versions; the column `dataset` says which. We keep the ones derived from GSM8K.

In [ ]:
step_dpo = load_dataset("xinlai/Math-Step-DPO-10K", split="train")
print(step_dpo)

gsm_pairs = step_dpo.filter(lambda example: "GSM" in example["dataset"])
print(f"{len(gsm_pairs):,} of {len(step_dpo):,} pairs come from GSM8K")

example = gsm_pairs[0]
for key in ["prompt", "initial_reason_steps", "full_chosen", "full_rejected", "answer"]:
    print(f"----- {key} -----\n{example[key]}\n")

**6.1 Complete `to_preference`**, which turns an example into TRL's conversational preference format: a `prompt` with the user message, and a `chosen` and a `rejected` assistant message. Both responses are the shared `initial_reason_steps` followed by `full_chosen` or `full_rejected`.

Hint: simply concatenate the strings; `full_chosen` and `full_rejected` already start with the right whitespace.

In [ ]:
def to_preference(example):
    # A Math-Step-DPO example -> {"prompt": [user], "chosen": [assistant], "rejected": [assistant]}
    return {
        "prompt": ...,    # TODO
        "chosen": ...,    # TODO
        "rejected": ...,  # TODO
    }


dpo_data = gsm_pairs.shuffle(seed=0).select(range(min(N_DPO, len(gsm_pairs))))
dpo_data = dpo_data.map(to_preference, remove_columns=gsm_pairs.column_names)
print(dpo_data)

In [ ]:
# ✅ Check your to_preference
converted = to_preference(example)
assert converted["prompt"] == [{"role": "user", "content": example["prompt"]}], "the prompt should be one user message"
for key in ["chosen", "rejected"]:
    assert len(converted[key]) == 1 and converted[key][0]["role"] == "assistant", f"{key} should be one assistant message"
    assert converted[key][0]["content"].startswith(example["initial_reason_steps"]), f"{key} should start with the shared steps"
assert converted["chosen"][0]["content"].endswith(example["full_chosen"]) and converted["rejected"][0]["content"].endswith(example["full_rejected"])
print("Looks good ✅")

**6.2 The chosen and the rejected response begin with exactly the same tokens** (the shared steps). What do these tokens contribute to the DPO loss? What does that mean for which part of the responses the model learns from?

Hint: $\log \pi(y \mid x)$ is the sum of the log-probabilities of the tokens of $y$, and every token is predicted from the tokens before it.

---

*Your answer here:*

---

### Training with DPO

We continue from the LoRA SFT model: we load the saved adapter on top of the base model, with its LoRA matrices trainable. DPO also needs the frozen reference model $\pi_\text{ref}$. With a PEFT model, `DPOTrainer` does not need a second copy of the whole model: it keeps a frozen copy of the *adapter* as the reference.

**6.3 Create the `DPOTrainer`** with the model, the configuration, the preference data and the tokenizer, and train. On a T4 this takes roughly 10–15 minutes: every step processes both responses with the model and with the reference.

In [ ]:
model = AutoPeftModelForCausalLM.from_pretrained(LORA_DIR, is_trainable=True, dtype=torch.float32).to(device)

dpo_config = DPOConfig(
    output_dir="qwen-gsm8k-dpo",
    beta=0.1,                            # strength of the pull towards the reference model
    num_train_epochs=1,
    per_device_train_batch_size=4,       # preference pairs per pass (each pair has two responses)
    gradient_accumulation_steps=4,       # an effective batch of 16 pairs
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=0.1,
    max_length=768,                      # prompt + response
    gradient_checkpointing=True,
    bf16=USE_BF16,
    fp16=USE_FP16,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    seed=0,
)

trainer = ...  # TODO: a DPOTrainer

start = time.time()
trainer.train()
print(f"Training took {time.time() - start:.0f} s")

During training, `DPOTrainer` logs the implicit rewards of the chosen and the rejected responses, their difference (the *margin*), and the fraction of pairs in which the chosen response has the higher reward (the *reward accuracy*).

In [ ]:
logs = [log for log in trainer.state.log_history if "rewards/margins" in log]
steps = [log["step"] for log in logs]

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for key in ["rewards/chosen", "rewards/rejected", "rewards/margins"]:
    axes[0].plot(steps, [log[key] for log in logs], label=key.split("/")[1])
axes[0].axhline(0, color="gray", linewidth=0.5)
axes[0].set_xlabel("training step")
axes[0].set_title("implicit rewards")
axes[0].legend()
axes[1].plot(steps, [log["rewards/accuracies"] for log in logs])
axes[1].set_ylim(0, 1.05)
axes[1].set_xlabel("training step")
axes[1].set_title("reward accuracy")
plt.tight_layout()
plt.show()

**6.4 Describe the curves.** What happens to the rewards of the chosen and of the rejected responses, and to the margin? Is it what you expected?

---

*Your answer here:*

---

In [ ]:
scores["LoRA SFT + DPO"], dpo_replies = evaluate(trainer.model, "LoRA SFT + DPO")
print(dpo_replies[0])

plt.figure(figsize=(7, 3))
plt.barh(list(scores), list(scores.values()))
plt.xlim(0, 1)
plt.xlabel(f"accuracy on {N_EVAL} GSM8K test problems")
plt.gca().invert_yaxis()
plt.show()

**6.5 Did DPO improve the accuracy?** Whatever your result: give reasons why preference alignment on 1,000 pairs might help only a little (or even hurt) in this setting, and say what DPO and RLHF are mainly used for in practice.

---

*Your answer here:*

---

## 7. MCQ

Answer each question by writing the letter of your choice (A–D) after **Answer:**.

---

### 7.1. The SFT objective

What is the training objective of supervised fine-tuning (SFT) of a decoder-only language model?

A. Masked language modelling on the prompts<br>
B. Next-token prediction (cross-entropy) on the responses of curated prompt–response pairs<br>
C. Maximising a reward given by a separate reward model<br>
D. Predicting which of two responses is better<br>

**Answer:** 

---

### 7.2. LoRA

Which statement about LoRA is correct?

A. It updates all weights, but stores them in 4-bit precision<br>
B. It freezes the pre-trained weights and trains a low-rank update $BA$ for selected weight matrices<br>
C. It removes the layers that are least important for the task<br>
D. It only trains the bias terms of the model<br>

**Answer:** 

---

### 7.3. Preference data

What does a training example for DPO consist of?

A. A prompt and a single correct response<br>
B. A prompt, a preferred response and a rejected response<br>
C. A response and a numerical reward between 0 and 1<br>
D. Two prompts and a label saying which one is harder<br>

**Answer:** 

---

### 7.4. The role of $\beta$ in DPO

What happens if you increase $\beta$ in the DPO loss?

A. The model is held closer to the reference model<br>
B. The model is allowed to move further away from the reference model<br>
C. The rejected responses are ignored<br>
D. The learning rate is increased<br>

**Answer:** 

---

### 7.5. RLHF vs. DPO

What is the main practical advantage of DPO over RLHF with PPO?

A. DPO does not need any preference data<br>
B. DPO always produces a better model<br>
C. DPO needs no separate reward model and no reinforcement-learning loop: it is a simple supervised loss on preference pairs<br>
D. DPO does not need a reference model<br>

**Answer:** 